# GraphicZero-HigherDegreedPs

Python migration of the legacy Zeppelin notebook **`GraphicZero-HigherDegreedP's`** (id `2G1TWRHEF`).

- Uses live Neo4j source logic equivalent to `DFScripts.s12QuadQ`.
- Preserves Zeppelin checkpoint naming through a `z` dictionary.
- Keeps legacy fold semantics (`degree=-1` -> `0`, `scalar*2`) and divisor `2` parity.


In [1]:
from __future__ import annotations

import math
from decimal import Decimal, InvalidOperation, getcontext
from pathlib import Path

import pandas as pd

getcontext().prec = 50

try:
    from neo4j import GraphDatabase
except ImportError as exc:
    raise ImportError("Install neo4j driver in this environment: pip install neo4j") from exc

z: dict[str, pd.DataFrame] = {}

RANGE_LOW = 2
RANGE_HIGH = 7
MAX_N = "8"


def parse_db_properties() -> dict[str, str]:
    candidates = [
        Path("ml/polys/db.properties"),
        Path("db.properties"),
        Path("../db.properties"),
    ]
    cfg_path = next((p for p in candidates if p.exists()), None)
    if cfg_path is None:
        raise FileNotFoundError("Could not find db.properties. Copy ml/polys/db.properties.example first.")

    props: dict[str, str] = {}
    for raw_line in cfg_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        props[key.strip()] = value.strip()

    required = ["neo4j.url", "neo4j.user", "neo4j.password", "neo4j.database"]
    missing = [k for k in required if not props.get(k)]
    if missing:
        raise ValueError(f"Missing db.properties keys: {missing}")

    raw_url = props["neo4j.url"].replace("jdbc:neo4j:", "")
    host_part = raw_url.split("//", 1)[-1]
    if ":" not in host_part:
        raw_url = f"{raw_url}:7687"
    props["bolt_url"] = raw_url
    return props


def load_s12_quadq_live(range_low: int, range_high: int, max_n: str) -> pd.DataFrame:
    cfg = parse_db_properties()
    cypher = """
    UNWIND range(toInteger($rangeLow), toInteger($rangeHigh)) AS n
    WITH toString(n) AS N, $maxN AS nMax
    MATCH (v:VertexNode)<-[]-(i:IndexedBy)-[]->(:Evaluate),
          (t:TwoSeqFactor)<-[]-(i)
    WHERE i.N = N AND i.MaxN = nMax AND i.Dimension = '2'
    RETURN i.N AS index,
           i.MaxN AS maxN,
           t.twoSeq AS rowScalar,
           '2' AS divisor,
           toString(CASE WHEN toString(v.Degree)='-1' THEN toInteger(v.Scalar) * 2 ELSE toInteger(v.Scalar) END) AS scalar,
           toString(CASE WHEN toString(v.Degree)='-1' THEN 0 ELSE toInteger(v.Degree) END) AS degree
    """
    driver = GraphDatabase.driver(
        cfg["bolt_url"],
        auth=(cfg["neo4j.user"], cfg["neo4j.password"]),
    )
    try:
        with driver.session(database=cfg["neo4j.database"]) as session:
            rows = [dict(record) for record in session.run(cypher, {"rangeLow": range_low, "rangeHigh": range_high, "maxN": max_n})]
    finally:
        driver.close()

    if not rows:
        raise ValueError("Live query returned no rows. Check Neo4j connection and input range.")

    df = pd.DataFrame(rows)
    expected_cols = ["index", "maxN", "rowScalar", "divisor", "scalar", "degree"]
    missing_cols = [c for c in expected_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Query result missing columns: {missing_cols}")

    for col in expected_cols:
        df[col] = df[col].astype(str)
    return df[expected_cols]


s12_live = load_s12_quadq_live(RANGE_LOW, RANGE_HIGH, MAX_N)
z["s12MaxN8"] = s12_live
print(f"Loaded s12MaxN8 rows={len(s12_live)}")
print(s12_live.head())


Loaded s12MaxN8 rows=114
  index maxN rowScalar divisor scalar degree
0     2    8         1       2      6      0
1     2    8         1       2     -6      1
2     2    8         1       2      1      2
3     2    8         1       2     42      0
4     2    8         1       2     -7      1


In [2]:
# === ResultUDF.main(Array("s12MaxN8","rUdfMaxN8")) ===
def extend_result_row(row: pd.Series) -> pd.Series:
    scalar = Decimal(str(row["scalar"]))
    row_scalar = Decimal(str(row["rowScalar"]))
    degree = int(str(row["degree"]))
    divisor = Decimal(str(row["divisor"]))
    max_n = Decimal(str(row["maxN"]))
    result = (scalar / divisor) * (max_n**degree) * row_scalar
    return pd.Series(
        {
            "dimension": "2",
            "degree": str(row["degree"]),
            "scalar": str(row["scalar"]),
            "index": str(row["index"]),
            "maxIndex": str(row["maxN"]),
            "divisor": str(row["divisor"]),
            "result": format(result, "f"),
            "rowScalar": str(row["rowScalar"]),
        }
    )


z["rUdfMaxN8"] = z["s12MaxN8"].apply(extend_result_row, axis=1)
print(f"rUdfMaxN8 rows={len(z['rUdfMaxN8'])}")
print(z["rUdfMaxN8"].head())


rUdfMaxN8 rows=114
  dimension degree scalar index maxIndex divisor result rowScalar
0         2      0      6     2        8       2      3         1
1         2      1     -6     2        8       2    -24         1
2         2      2      1     2        8       2   32.0         1
3         2      0     42     2        8       2     21         1
4         2      1     -7     2        8       2  -28.0         1


In [3]:
# === GroupedScalarRowsSum.main(Array("rUdfMaxN8","KvDS")) ===
ds = z["rUdfMaxN8"].copy()
ds["index_i"] = ds["index"].astype(int)
ds["rowScalar_i"] = ds["rowScalar"].astype(int)
ds["degree_i"] = ds["degree"].astype(int)
ds["scalar_i"] = ds["scalar"].astype(int)
kv = ds.groupby(["index_i", "rowScalar_i", "degree_i"], as_index=False)["scalar_i"].sum()
kv = kv.rename(
    columns={"index_i": "_1", "rowScalar_i": "_2", "degree_i": "_3", "scalar_i": "scalar_Result"}
)
kv["scalar_Result"] = kv["scalar_Result"].astype(str)
z["KvDS"] = kv
print(z["KvDS"].head())


   _1  _2  _3 scalar_Result
0   2   1   0           104
1   2   1   1           -28
2   2   1   2             2
3   3   1   0            80
4   3   1   1           -24


In [4]:
# === MapKVUDF.main(Array("KvDS","KvDSUnpavked")) ===
z["KvDSUnpavked"] = z["KvDS"].copy()
print(z["KvDSUnpavked"].head())


   _1  _2  _3 scalar_Result
0   2   1   0           104
1   2   1   1           -28
2   2   1   2             2
3   3   1   0            80
4   3   1   1           -24


In [5]:
# === RowScalarDegreePivot.main(Array("KvDSUnpavked","pivotDF")) ===
df = z["KvDSUnpavked"].copy()
df["N"] = df["_1"].astype(int)
df["rowScalar"] = df["_2"].astype(int)
df["degree"] = df["_3"].astype(int)
df["scalar"] = df["scalar_Result"].astype(int)
pivot_df = (
    df.pivot_table(index=["N", "rowScalar"], columns="degree", values="scalar", aggfunc="sum")
    .sort_index()
    .reset_index()
)
for c in (0, 1, 2):
    if c not in pivot_df.columns:
        pivot_df[c] = 0
pivot_df = pivot_df[["N", "rowScalar", 0, 1, 2]]
z["pivotDF"] = pivot_df
print(z["pivotDF"].head())


degree  N  rowScalar    0   1  2
0       2          1  104 -28  2
1       3          1   80 -24  2
2       3          2   56 -15  1
3       4          1   60 -20  2
4       4          2   42 -13  1


In [6]:
# === PivotEvaluateUDF.main(Array("pivotDF","RootsMaxn8Range")) ===
def to_decimal(value: object) -> Decimal:
    if pd.isna(value):
        return Decimal(0)
    try:
        return Decimal(str(value))
    except InvalidOperation:
        return Decimal(0)


def quad_equ(row: pd.Series) -> dict[str, str]:
    four = Decimal(4)
    two = Decimal(2)
    zero = Decimal(0)
    row_scalar = to_decimal(row["rowScalar"])
    a = to_decimal(row[2])
    b = to_decimal(row[1])
    c = to_decimal(row[0])
    a_m = (row_scalar * a) / two
    b_m = (row_scalar * b) / two
    c_m = (row_scalar * c) / two
    two_a = a_m * two
    neg_bd_two_a = (zero - b_m) / two_a if two_a != 0 else Decimal(0)
    disc = (b_m**2) - (four * a_m * c_m)
    if disc > 0 and two_a != 0:
        disc_sqrt = Decimal(str(math.sqrt(float(disc))))
        root_1 = neg_bd_two_a + (disc_sqrt / two_a)
        root_2 = neg_bd_two_a - (disc_sqrt / two_a)
        return {
            "N": str(int(row["N"])),
            "rowScalar": str(int(row["rowScalar"])),
            "root1": format(root_1, "f"),
            "root2": format(root_2, "f"),
            "a": format(a, "f"),
            "b": format(b, "f"),
            "c": format(c, "f"),
        }
    return {
        "N": str(int(row["N"])),
        "rowScalar": str(int(row["rowScalar"])),
        "root1": "no",
        "root2": "root",
        "a": format(a, "f"),
        "b": format(b, "f"),
        "c": format(c, "f"),
    }


z["RootsMaxn8Range"] = pd.DataFrame([quad_equ(r) for _, r in z["pivotDF"].iterrows()])
print(z["RootsMaxn8Range"].head())


   N rowScalar root1 root2  a    b    c
0  2         1    no  root  2  -28  104
1  3         1    no  root  2  -24   80
2  3         2   8.0   7.0  1  -15   56
3  4         1    no  root  2  -20   60
4  4         2   7.0   6.0  1  -13   42


In [7]:
# === GroupedDegreeSum.main(Array("rUdfMaxN8","KvDegreeDS")) ===
ds2 = z["rUdfMaxN8"].copy()
ds2["weighted"] = ds2.apply(
    lambda r: Decimal(str(r["scalar"])) * Decimal(str(r["rowScalar"])) / Decimal(str(r["divisor"])),
    axis=1,
)
ds2["maxIndex_i"] = ds2["maxIndex"].astype(int)
ds2["index_i"] = ds2["index"].astype(int)
ds2["degree_i"] = ds2["degree"].astype(int)
gds = ds2.groupby(["maxIndex_i", "index_i", "degree_i"], as_index=False)["weighted"].sum()
gds = gds.rename(
    columns={"maxIndex_i": "_1", "index_i": "_2", "degree_i": "_3", "weighted": "scalar_Result"}
)
gds["scalar_Result"] = gds["scalar_Result"].map(lambda v: format(v, "f"))
z["KvDegreeDS"] = gds
z["KvDegreeDSUnpavked"] = gds.copy()
print(z["KvDegreeDS"].head())


   _1  _2  _3 scalar_Result
0   8   2   0            52
1   8   2   1         -14.0
2   8   2   2           1.0
3   8   3   0            96
4   8   3   1         -27.0


In [8]:
# === DropKeyColumn -> ReducedMaxN8Range + AggrigatedScalarDegreePivot ===
df2 = z["KvDegreeDSUnpavked"].copy()
reduced = pd.DataFrame(
    {
        "MaxN": df2["_1"].astype(str),
        "N": df2["_2"].astype(str),
        "Degree": df2["_3"].astype(str),
        "Scalar": df2["scalar_Result"].astype(str),
    }
)
z["ReducedMaxN8Range"] = reduced

agg = reduced.copy()
agg["Scalar_d"] = agg["Scalar"].map(lambda v: Decimal(str(v)))
pivot21 = agg.pivot_table(index="N", columns="Degree", values="Scalar_d", aggfunc="sum")
for c in ("0", "1", "2"):
    if c not in pivot21.columns:
        pivot21[c] = Decimal(0)
pivot21 = pivot21[["0", "1", "2"]].reset_index()
z["ReducedMaxN8RangePivot"] = pivot21
print(z["ReducedMaxN8Range"].head())
print(z["ReducedMaxN8RangePivot"].head())


  MaxN  N Degree Scalar
0    8  2      0     52
1    8  2      1  -14.0
2    8  2      2    1.0
3    8  3      0     96
4    8  3      1  -27.0
Degree  N    0       1     2
0       2   52   -14.0   1.0
1       3   96   -27.0   2.0
2       4  184   -53.0   4.0
3       5  360  -105.0   8.0
4       6  712  -209.0  16.0


In [9]:
# === ReducedResultUDF + ReducedResultUDFPivot + ReducedRowsSum ===
rr = z["ReducedMaxN8Range"].copy()
rows = []
for _, r in rr.iterrows():
    scalar = Decimal(str(r["Scalar"]))
    max_n = int(str(r["MaxN"]))
    degree = int(str(r["Degree"]))
    result = scalar * (Decimal(max_n) ** degree)
    rows.append(
        {
            "Scalar": str(r["Scalar"]),
            "N": str(r["N"]),
            "Degree": str(r["Degree"]),
            "result": format(result, "f"),
            "MaxN": str(r["MaxN"]),
        }
    )
res_df = pd.DataFrame(rows)
z["ReducedMaxN8RangeResult"] = res_df

res_df["result_d"] = res_df["result"].map(lambda v: Decimal(str(v)))
res_pivot = res_df.pivot_table(index=["MaxN", "N"], columns="Degree", values="result_d", aggfunc="sum")
for c in ("0", "1", "2"):
    if c not in res_pivot.columns:
        res_pivot[c] = Decimal(0)
res_pivot = res_pivot[["0", "1", "2"]].reset_index()
z["ReducedMaxN8RangeResultPivot"] = res_pivot

final = res_df.groupby(["MaxN", "N"], as_index=False)["result_d"].sum()
final = final.rename(columns={"result_d": "index_Result"})
final["index_Result"] = final["index_Result"].map(lambda v: format(v, "f"))
z["ReducedMaxN8RangeFinalResult"] = final

print(z["ReducedMaxN8RangeResult"].head())
print(z["ReducedMaxN8RangeResultPivot"].head())
print(z["ReducedMaxN8RangeFinalResult"].head())


  Scalar  N Degree  result MaxN result_d
0     52  2      0      52    8       52
1  -14.0  2      1  -112.0    8   -112.0
2    1.0  2      2    64.0    8     64.0
3     96  3      0      96    8       96
4  -27.0  3      1  -216.0    8   -216.0
Degree MaxN  N    0        1       2
0         8  2   52   -112.0    64.0
1         8  3   96   -216.0   128.0
2         8  4  184   -424.0   256.0
3         8  5  360   -840.0   512.0
4         8  6  712  -1672.0  1024.0
  MaxN  N index_Result
0    8  2          4.0
1    8  3          8.0
2    8  4         16.0
3    8  5         32.0
4    8  6         64.0


In [10]:
# === FilterRowScalar / showcase cells ===
base = z["rUdfMaxN8"].copy()
f1 = base[(base["index"].astype(str) == "7") & (base["rowScalar"].astype(str) == "1")].reset_index(drop=True)
z["index7MaxN8RowScalar1"] = f1

print("RootsMaxn8Range")
print(z["RootsMaxn8Range"].to_string(index=False))
print("\nReducedMaxN8RangePivot")
print(z["ReducedMaxN8RangePivot"].to_string(index=False))
print("\nReducedMaxN8RangeResultPivot")
print(z["ReducedMaxN8RangeResultPivot"].to_string(index=False))
print("\nReducedMaxN8RangeFinalResult")
print(z["ReducedMaxN8RangeFinalResult"].to_string(index=False))
print("\nindex7MaxN8RowScalar1")
print(z["index7MaxN8RowScalar1"].to_string(index=False))


RootsMaxn8Range
N rowScalar root1 root2 a   b   c
2         1    no  root 2 -28 104
3         1    no  root 2 -24  80
3         2   8.0   7.0 1 -15  56
4         1    no  root 2 -20  60
4         2   7.0   6.0 1 -13  42
4         4   8.0   7.0 1 -15  56
5         1    no  root 2 -16  44
5         2   6.0   5.0 1 -11  30
5         4   7.0   6.0 1 -13  42
5         8   8.0   7.0 1 -15  56
6         1    no  root 2 -12  32
6         2   5.0   4.0 1  -9  20
6         4   6.0   5.0 1 -11  30
6         8   7.0   6.0 1 -13  42
6        16   8.0   7.0 1 -15  56
7         1    no  root 2  -8  24
7         2   4.0   3.0 1  -7  12
7         4   5.0   4.0 1  -9  20
7         8   6.0   5.0 1 -11  30
7        16   7.0   6.0 1 -13  42
7        32   8.0   7.0 1 -15  56

ReducedMaxN8RangePivot
N    0      1    2
2   52  -14.0  1.0
3   96  -27.0  2.0
4  184  -53.0  4.0
5  360 -105.0  8.0
6  712 -209.0 16.0
7 1416 -417.0 32.0

ReducedMaxN8RangeResultPivot
MaxN N    0       1      2
   8 2   52  -112.0   

In [11]:
# === Stage-level parity checks ===
required_checkpoints = [
    "s12MaxN8",
    "rUdfMaxN8",
    "KvDS",
    "KvDSUnpavked",
    "pivotDF",
    "RootsMaxn8Range",
    "KvDegreeDS",
    "KvDegreeDSUnpavked",
    "ReducedMaxN8Range",
    "ReducedMaxN8RangePivot",
    "ReducedMaxN8RangeResult",
    "ReducedMaxN8RangeResultPivot",
    "ReducedMaxN8RangeFinalResult",
    "index7MaxN8RowScalar1",
]

missing = [name for name in required_checkpoints if name not in z]
if missing:
    raise AssertionError(f"Missing checkpoints: {missing}")

for name in required_checkpoints:
    if z[name].empty:
        raise AssertionError(f"Checkpoint is empty: {name}")

expected_root_cols = {"N", "rowScalar", "root1", "root2", "a", "b", "c"}
if set(z["RootsMaxn8Range"].columns) != expected_root_cols:
    raise AssertionError("RootsMaxn8Range schema drift detected.")

expected_final_cols = {"MaxN", "N", "index_Result"}
if set(z["ReducedMaxN8RangeFinalResult"].columns) != expected_final_cols:
    raise AssertionError("ReducedMaxN8RangeFinalResult schema drift detected.")

print("All stage checks passed.")
print({name: len(z[name]) for name in required_checkpoints})


All stage checks passed.
{'s12MaxN8': 114, 'rUdfMaxN8': 114, 'KvDS': 63, 'KvDSUnpavked': 63, 'pivotDF': 21, 'RootsMaxn8Range': 21, 'KvDegreeDS': 18, 'KvDegreeDSUnpavked': 18, 'ReducedMaxN8Range': 18, 'ReducedMaxN8RangePivot': 6, 'ReducedMaxN8RangeResult': 18, 'ReducedMaxN8RangeResultPivot': 6, 'ReducedMaxN8RangeFinalResult': 6, 'index7MaxN8RowScalar1': 9}
